# Node Transformer + BERT Sentiment Fusion — Stock Price Forecasting

Based on: Al Ridhawi, Haj Ali, Al Osman, *"Stock Market Prediction Using Node Transformer Architecture Integrated with BERT Sentiment Analysis"* ([arXiv:2603.05917](https://arxiv.org/abs/2603.05917)).

The paper models the market as a graph — stocks are nodes, edges are sector/supply-chain relationships — and combines:

1. A **Node Transformer**: temporal attention per stock (price/volume history) + cross-sectional graph attention across related stocks
2. **BERT sentiment** (FinBERT): sentiment extracted from headlines/social text, fused in via attention rather than just concatenated

Output: next-day return/price forecast per ticker with a directional-accuracy estimate. Reported paper metrics: 0.80% MAPE vs. 1.20% (ARIMA) / 1.00% (LSTM), sentiment reducing error ~10% overall and ~25% around earnings, trained on 20 S&P 500 names, Jan 1982–Mar 2025.

**This notebook reproduces the architecture, not the paper's exact reported result.** Validate on your own data before trusting any number out of it.

**Where this fits in the app**: this is a *supplementary* forecast signal for the Python service described in `../tradingskills.md` (§6). It feeds into Alpha's report card alongside the SMC/Liquidity/Composite skills — it is not a standalone trade trigger, and per §13 of that doc it never authorizes auto-execution.

A second reference the user pointed to — an SSRN paper on real-time transformer market forecasting — was blocked by bot-protection when fetched, so its specifics aren't reflected here. Revisit if you can get authenticated access; it's likely relevant to the streaming/low-latency inference story (see §10 caveats).

## 0. Requirements

```bash
pip install torch transformers yfinance pandas numpy scikit-learn requests
```

GPU is optional — the model here is small enough to train on CPU/MPS for a handful of tickers, but expect it to be slow for large universes.

**Finnhub API key** (for the headline corpus in §4): sign up free at [finnhub.io/register](https://finnhub.io/register), then set it as an environment variable rather than pasting it into the notebook:

```bash
export FINNHUB_API_KEY="your-key-here"
```

In [ ]:
import math
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {DEVICE}")

## 1. Config — tickers, date range, sector map (graph edges)

`LOOKBACK` defaults to 60 bars to match the per-decision data budget in `tradingskills.md` §11 — training itself uses the full historical range below (that's the explicit, separate long-history budget), but each individual training/inference *sample* only looks back 60 bars, same as production inference will.

In [ ]:
@dataclass
class Config:
    tickers: list = field(default_factory=lambda: [
        "AAPL", "MSFT", "NVDA", "AMD", "AVGO",   # tech / semis
        "JPM", "BAC", "GS",                        # financials
        "XOM", "CVX",                               # energy
    ])
    # Simple sector proxy for graph edges. Replace with GICS sector/supply-chain
    # data for a closer reproduction of the paper's graph construction.
    sector_map: dict = field(default_factory=lambda: {
        "AAPL": "tech", "MSFT": "tech", "NVDA": "semis", "AMD": "semis", "AVGO": "semis",
        "JPM": "financials", "BAC": "financials", "GS": "financials",
        "XOM": "energy", "CVX": "energy",
    })
    start_date: str = "2018-01-01"
    end_date: str = "2025-01-01"
    lookback: int = 60       # bars per training/inference sample (matches tradingskills.md default)
    horizon: int = 1         # next-day forecast
    train_frac: float = 0.7
    val_frac: float = 0.15   # remainder is test
    batch_size: int = 32
    epochs: int = 25
    lr: float = 1e-4
    d_model: int = 64
    n_heads: int = 4
    n_temporal_layers: int = 2

cfg = Config()
cfg

## 2. Data ingestion — historical OHLCV

Uses `yfinance` (no API key). Computes a small technical feature set per bar — return, log-return, rolling volatility, and a simple RSI — which becomes the Node Transformer's per-node input.

In [ ]:
# price_frames = load_price_data(cfg.tickers, cfg.start_date, cfg.end_date)
#
# headline_source = build_headline_corpus(cfg.tickers, cfg.start_date, cfg.end_date)
# dates, X, S, y = build_panel(price_frames, cfg.tickers, headline_source=headline_source)
#
# train_idx, val_idx, test_idx = chronological_split(len(dates), cfg.train_frac, cfg.val_frac)
# train_ds = NodeWindowDataset(X[train_idx], S[train_idx], y[train_idx], cfg.lookback)
# val_ds = NodeWindowDataset(X[val_idx], S[val_idx], y[val_idx], cfg.lookback)
# test_ds = NodeWindowDataset(X[test_idx], S[test_idx], y[test_idx], cfg.lookback)
#
# train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
# test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)
#
# model = NodeTransformerForecaster(
#     n_nodes=len(cfg.tickers), in_dim=len(FEATURE_COLS), lookback=cfg.lookback,
#     d_model=cfg.d_model, n_heads=cfg.n_heads, n_temporal_layers=cfg.n_temporal_layers,
# )
# model = train_model(model, train_loader, val_loader, adjacency, cfg)
# print(evaluate(model, test_loader, adjacency))

## 3. Graph construction — nodes = tickers, edges = sector adjacency

Same-sector tickers get an edge; every node also gets a self-loop. Normalized the way a GCN normalizes adjacency (`D^-1/2 (A+I) D^-1/2`) so the graph attention layer's mask is well-scaled.

In [ ]:
def build_adjacency(tickers: list, sector_map: dict) -> torch.Tensor:
    n = len(tickers)
    A = np.eye(n)  # self-loops
    for i, ti in enumerate(tickers):
        for j, tj in enumerate(tickers):
            if i != j and sector_map.get(ti) == sector_map.get(tj):
                A[i, j] = 1.0
    deg = A.sum(axis=1)
    d_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
    D_inv_sqrt = np.diag(d_inv_sqrt)
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return torch.tensor(A_norm, dtype=torch.float32)


adjacency = build_adjacency(cfg.tickers, cfg.sector_map)
adjacency.shape

## 4. Sentiment pipeline — Finnhub headlines + FinBERT scoring

Two steps: (4.1) pull and cache historical headlines per ticker from Finnhub — this builds the actual **training corpus**, done once and reused across training runs; (4.2) score cached headlines with a pretrained FinBERT (`ProsusAI/finbert`) — no fine-tuning required to get a usable signal.

Finnhub was chosen over Alpha Vantage (25 requests/day free tier — too slow for bulk historical backfill across a watchlist) and Stocktwits (its public stream endpoint is oriented at recent/live messages, not a deep historical archive) specifically because building years of training data needs real throughput. Its free tier's 60 requests/min makes that practical; verify its news coverage actually reaches back as far as your `start_date` before committing to a full backfill — coverage for older, less-liquid tickers isn't guaranteed.

### 4.1 Historical headline corpus — Finnhub

Pulls `/company-news` per ticker, chunked by month, and caches everything to disk (`data/finnhub_news_cache/<TICKER>.json`) so re-running the notebook — or retraining later — never re-hits the API for a date range it already has. Rate-limited to stay safely under the free tier's 60 requests/minute, with a one-time backoff-and-retry on HTTP 429.

In [ ]:
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

import requests

FINNHUB_NEWS_URL = "https://finnhub.io/api/v1/company-news"
FINNHUB_SAFE_REQUESTS_PER_MIN = 55  # stay under the 60/min free-tier cap with headroom
NEWS_CACHE_DIR = Path("data/finnhub_news_cache")


def _month_chunks(start: str, end: str):
    """Yields (chunk_start, chunk_end) ~month-sized date strings between start and end."""
    cur = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    while cur < end_dt:
        chunk_end = min(cur + timedelta(days=30), end_dt)
        yield cur.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")
        cur = chunk_end + timedelta(days=1)


def _load_cache(cache_file: Path) -> dict:
    return json.loads(cache_file.read_text()) if cache_file.exists() else {"_fetched_chunks": []}


def fetch_finnhub_news(ticker: str, start: str, end: str, api_key: str,
                        cache_dir: Path = NEWS_CACHE_DIR,
                        requests_per_minute: int = FINNHUB_SAFE_REQUESTS_PER_MIN) -> dict:
    """Returns {date_str: [headline, ...]} for `ticker` between start/end, backed by an
    on-disk cache so previously-fetched month chunks are never re-requested.
    """
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f"{ticker}.json"
    cached = _load_cache(cache_file)
    min_interval = 60.0 / requests_per_minute

    for chunk_start, chunk_end in _month_chunks(start, end):
        chunk_key = f"{chunk_start}_{chunk_end}"
        if chunk_key in cached["_fetched_chunks"]:
            continue

        params = {"symbol": ticker, "from": chunk_start, "to": chunk_end, "token": api_key}
        resp = requests.get(FINNHUB_NEWS_URL, params=params, timeout=15)
        if resp.status_code == 429:
            time.sleep(60)  # one-time backoff, then retry
            resp = requests.get(FINNHUB_NEWS_URL, params=params, timeout=15)
        resp.raise_for_status()

        for article in resp.json():
            date_str = datetime.utcfromtimestamp(article["datetime"]).strftime("%Y-%m-%d")
            headline = article.get("headline", "").strip()
            if not headline:
                continue
            cached.setdefault(date_str, [])
            if headline not in cached[date_str]:
                cached[date_str].append(headline)

        cached["_fetched_chunks"].append(chunk_key)
        cache_file.write_text(json.dumps(cached))
        time.sleep(min_interval)

    return {k: v for k, v in cached.items() if k != "_fetched_chunks"}


def build_headline_corpus(tickers: list, start: str, end: str, api_key: Optional[str] = None,
                           cache_dir: Path = NEWS_CACHE_DIR) -> dict:
    """Builds the {(ticker, date_str): [headlines]} lookup `daily_sentiment_scores` /
    `build_panel` expect below. Safe to re-run — cached ticker/date-range chunks are
    skipped, so a second run only fetches whatever's new.
    """
    api_key = api_key or os.environ.get("FINNHUB_API_KEY")
    if not api_key:
        raise ValueError(
            "Set FINNHUB_API_KEY (free tier at finnhub.io/register) or pass api_key= explicitly."
        )

    corpus = {}
    for ticker in tickers:
        for date_str, headlines in fetch_finnhub_news(ticker, start, end, api_key, cache_dir).items():
            corpus[(ticker, date_str)] = headlines
    return corpus


# headline_source = build_headline_corpus(cfg.tickers, cfg.start_date, cfg.end_date)
# len(headline_source)  # number of (ticker, date) pairs with at least one cached headline

### 4.2 FinBERT sentiment scoring

Scores whatever headlines came out of §4.1 (or any other `{(ticker, date_str): [headlines]}` source you point it at) with a pretrained FinBERT — no fine-tuning needed to get a usable signal; fine-tune later on labeled financial text if you have it, per the paper's approach. Falls back to a neutral 0.0 for any (ticker, date) with no cached headline, so the rest of the notebook still runs end-to-end even before the corpus is fully built.

In [ ]:
_sentiment_pipeline = None

def get_sentiment_pipeline():
    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        from transformers import pipeline
        _sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    return _sentiment_pipeline


def score_headlines(headlines: list) -> float:
    """Returns a single signed sentiment score in [-1, 1] (positive_prob - negative_prob)."""
    if not headlines:
        return 0.0
    clf = get_sentiment_pipeline()
    results = clf(headlines, truncation=True)
    score = 0.0
    for r in results:
        sign = {"positive": 1, "negative": -1, "neutral": 0}[r["label"].lower()]
        score += sign * r["score"]
    return score / len(results)


def daily_sentiment_scores(ticker: str, dates: pd.DatetimeIndex, headline_source: Optional[dict] = None) -> pd.Series:
    """headline_source: optional {(ticker, date_str): [headlines]} lookup.

    IMPORTANT: only ever use headlines with a published timestamp strictly
    before the bar's close — using same-day-after-close or future headlines
    here is a lookahead-bias bug, not a feature.
    """
    scores = []
    for d in dates:
        key = (ticker, d.strftime("%Y-%m-%d"))
        headlines = (headline_source or {}).get(key, [])
        scores.append(score_headlines(headlines) if headlines else 0.0)
    return pd.Series(scores, index=dates, name="sentiment")

## 5. Dataset construction — sliding windows across all nodes, chronological split

Split is time-ordered (train → val → test), never shuffled across the boundary — a random split would leak future information into training, which is the single easiest way to fool yourself with a return-prediction backtest.

In [ ]:
FEATURE_COLS = ["return", "log_return", "vol_20", "rsi_14"]


def build_panel(price_frames: dict, tickers: list, headline_source: Optional[dict] = None) -> tuple:
    """Aligns all tickers on a common date index and stacks features + sentiment.

    Returns:
        dates: aligned DatetimeIndex
        X: (T, N, F) technical features
        S: (T, N, 1) sentiment scores
        y: (T, N) next-bar return (the prediction target)
    """
    common_index = None
    for t in tickers:
        idx = price_frames[t].index
        common_index = idx if common_index is None else common_index.intersection(idx)
    common_index = common_index.sort_values()

    X_list, S_list, y_list = [], [], []
    for t in tickers:
        df = price_frames[t].loc[common_index]
        X_list.append(df[FEATURE_COLS].values)
        sent = daily_sentiment_scores(t, common_index, headline_source)
        S_list.append(sent.values.reshape(-1, 1))
        y_list.append(df["return"].shift(-1).values)  # next-day return target

    X = np.stack(X_list, axis=1)  # (T, N, F)
    S = np.stack(S_list, axis=1)  # (T, N, 1)
    y = np.stack(y_list, axis=1)  # (T, N)

    valid = ~np.isnan(y).any(axis=1)
    return common_index[valid], X[valid], S[valid], y[valid]


class NodeWindowDataset(Dataset):
    def __init__(self, X: np.ndarray, S: np.ndarray, y: np.ndarray, lookback: int):
        self.X, self.S, self.y, self.lookback = X, S, y, lookback

    def __len__(self):
        return len(self.X) - self.lookback

    def __getitem__(self, idx):
        x_window = self.X[idx: idx + self.lookback]        # (lookback, N, F)
        s_window = self.S[idx: idx + self.lookback]        # (lookback, N, 1)
        target = self.y[idx + self.lookback - 1]            # (N,) next-day return as of window end
        return (
            torch.tensor(x_window, dtype=torch.float32),
            torch.tensor(s_window, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32),
        )


def chronological_split(n: int, train_frac: float, val_frac: float) -> tuple:
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    return slice(0, train_end), slice(train_end, val_end), slice(val_end, n)

## 6. Model — Node Transformer + Sentiment Fusion

Three pieces, matching the paper's description:

1. **Temporal encoder**: a standard Transformer encoder applied along the time axis, independently per node — captures each stock's own history.
2. **Graph attention layer**: cross-sectional multi-head attention across nodes at the final timestep, masked by the sector adjacency matrix — captures cross-stock dependencies (sector co-movement etc).
3. **Sentiment fusion**: the sentiment window is separately encoded and fused into the node embedding via an attention-gated combination before the regression head, rather than plain concatenation.

In [ ]:
class TemporalEncoder(nn.Module):
    """Per-node transformer over the lookback window."""

    def __init__(self, in_dim: int, d_model: int, n_heads: int, n_layers: int, lookback: int):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, lookback, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            batch_first=True, dropout=0.1,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        # x: (B, L, F) for a single node -> (B, L, d_model)
        h = self.input_proj(x) + self.pos_embedding[:, : x.size(1)]
        h = self.encoder(h)
        return h[:, -1, :]  # embedding at the final timestep


class GraphAttentionLayer(nn.Module):
    """Cross-sectional attention across nodes, masked by the sector adjacency matrix."""

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, node_embeddings, adjacency):
        # node_embeddings: (B, N, d_model); adjacency: (N, N)
        mask = (adjacency == 0)  # True = block attention between unrelated nodes
        attn_out, _ = self.attn(node_embeddings, node_embeddings, node_embeddings, attn_mask=mask)
        return self.norm(node_embeddings + attn_out)


class SentimentFusion(nn.Module):
    """Encodes the sentiment window and gates it into the node embedding."""

    def __init__(self, d_model: int, lookback: int):
        super().__init__()
        self.sentiment_proj = nn.Sequential(
            nn.Linear(lookback, d_model), nn.ReLU(), nn.Linear(d_model, d_model),
        )
        self.gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())

    def forward(self, node_embedding, sentiment_window):
        # node_embedding: (B, d_model); sentiment_window: (B, L)
        s_embed = self.sentiment_proj(sentiment_window)
        gate = self.gate(torch.cat([node_embedding, s_embed], dim=-1))
        return node_embedding + gate * s_embed


class NodeTransformerForecaster(nn.Module):
    def __init__(self, n_nodes: int, in_dim: int, lookback: int, d_model: int, n_heads: int, n_temporal_layers: int):
        super().__init__()
        self.n_nodes = n_nodes
        self.temporal_encoder = TemporalEncoder(in_dim, d_model, n_heads, n_temporal_layers, lookback)
        self.graph_attention = GraphAttentionLayer(d_model, n_heads)
        self.sentiment_fusion = SentimentFusion(d_model, lookback)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Linear(d_model // 2, 1),
        )

    def forward(self, x, sentiment, adjacency):
        # x: (B, L, N, F); sentiment: (B, L, N, 1)
        B, L, N, Fdim = x.shape
        x_per_node = x.permute(0, 2, 1, 3).reshape(B * N, L, Fdim)          # (B*N, L, F)
        node_embeddings = self.temporal_encoder(x_per_node).view(B, N, -1)  # (B, N, d_model)

        node_embeddings = self.graph_attention(node_embeddings, adjacency)  # (B, N, d_model)

        sentiment_per_node = sentiment.permute(0, 2, 1, 3).reshape(B * N, L)  # (B*N, L)
        fused = self.sentiment_fusion(
            node_embeddings.reshape(B * N, -1), sentiment_per_node
        ).view(B, N, -1)

        return self.head(fused).squeeze(-1)  # (B, N) predicted next-day return per node

## 7. Training loop — walk-forward, MAPE-aware loss

In [ ]:
def mape_loss(pred, target, eps: float = 1e-4):
    return torch.mean(torch.abs((target - pred) / (target.abs() + eps)))


def train_model(model, train_loader, val_loader, adjacency, cfg: Config, patience: int = 5):
    model.to(DEVICE)
    adjacency = adjacency.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    best_val_loss, epochs_no_improve, best_state = float("inf"), 0, None

    for epoch in range(cfg.epochs):
        model.train()
        train_losses = []
        for x, s, y in train_loader:
            x, s, y = x.to(DEVICE), s.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(x, s, adjacency)
            loss = F.smooth_l1_loss(pred, y) + 0.1 * mape_loss(pred, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x, s, y in val_loader:
                x, s, y = x.to(DEVICE), s.to(DEVICE), y.to(DEVICE)
                pred = model(x, s, adjacency)
                val_losses.append(F.smooth_l1_loss(pred, y).item())

        val_loss = float(np.mean(val_losses))
        print(f"epoch {epoch+1:02d} | train_loss={np.mean(train_losses):.5f} | val_loss={val_loss:.5f}")

        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve, best_state = val_loss, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## 8. Evaluation — MAPE, directional accuracy, naive baseline

Compares against a naive "predict zero return" baseline. Swap in `statsmodels.tsa.arima.model.ARIMA` per-ticker and a plain LSTM if you want the same baselines the paper reports against.

In [ ]:
def evaluate(model, loader, adjacency):
    model.eval()
    adjacency = adjacency.to(DEVICE)
    preds, targets = [], []
    with torch.no_grad():
        for x, s, y in loader:
            x, s, y = x.to(DEVICE), s.to(DEVICE), y.to(DEVICE)
            pred = model(x, s, adjacency)
            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)

    mape = np.mean(np.abs((targets - preds) / (np.abs(targets) + 1e-4))) * 100
    directional_acc = np.mean(np.sign(preds) == np.sign(targets))
    naive_mape = np.mean(np.abs(targets) / (np.abs(targets) + 1e-4)) * 100  # baseline: predict 0

    return {
        "mape_pct": round(float(mape), 3),
        "directional_accuracy": round(float(directional_acc), 3),
        "naive_baseline_mape_pct": round(float(naive_mape), 3),
    }

## 9. Putting it together — run end-to-end

Uncomment to actually pull data and train. Left commented by default since it downloads market data and a ~400MB FinBERT checkpoint on first run.

In [ ]:
# price_frames = load_price_data(cfg.tickers, cfg.start_date, cfg.end_date)
# dates, X, S, y = build_panel(price_frames, cfg.tickers)  # headline_source=None -> neutral sentiment
#
# train_idx, val_idx, test_idx = chronological_split(len(dates), cfg.train_frac, cfg.val_frac)
# train_ds = NodeWindowDataset(X[train_idx], S[train_idx], y[train_idx], cfg.lookback)
# val_ds = NodeWindowDataset(X[val_idx], S[val_idx], y[val_idx], cfg.lookback)
# test_ds = NodeWindowDataset(X[test_idx], S[test_idx], y[test_idx], cfg.lookback)
#
# train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
# test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)
#
# model = NodeTransformerForecaster(
#     n_nodes=len(cfg.tickers), in_dim=len(FEATURE_COLS), lookback=cfg.lookback,
#     d_model=cfg.d_model, n_heads=cfg.n_heads, n_temporal_layers=cfg.n_temporal_layers,
# )
# model = train_model(model, train_loader, val_loader, adjacency, cfg)
# print(evaluate(model, test_loader, adjacency))

## 10. Export for the Python service

Saves everything the inference wrapper needs: model weights, the config, the ticker list/order, and the adjacency matrix (so the service doesn't need to rebuild the sector graph at inference time).

In [ ]:
import json
from pathlib import Path

ARTIFACT_DIR = Path("artifacts")


def export_model(model, adjacency, cfg: Config, out_dir: Path = ARTIFACT_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / "node_transformer_forecaster.pt")
    torch.save(adjacency, out_dir / "adjacency.pt")
    with open(out_dir / "config.json", "w") as f:
        json.dump({
            "tickers": cfg.tickers, "lookback": cfg.lookback, "d_model": cfg.d_model,
            "n_heads": cfg.n_heads, "n_temporal_layers": cfg.n_temporal_layers,
            "feature_cols": FEATURE_COLS,
        }, f, indent=2)
    print(f"Saved model artifacts to {out_dir.resolve()}")


def load_model_for_inference(artifact_dir: Path = ARTIFACT_DIR) -> tuple:
    with open(artifact_dir / "config.json") as f:
        saved_cfg = json.load(f)
    model = NodeTransformerForecaster(
        n_nodes=len(saved_cfg["tickers"]), in_dim=len(saved_cfg["feature_cols"]),
        lookback=saved_cfg["lookback"], d_model=saved_cfg["d_model"],
        n_heads=saved_cfg["n_heads"], n_temporal_layers=saved_cfg["n_temporal_layers"],
    )
    model.load_state_dict(torch.load(artifact_dir / "node_transformer_forecaster.pt", map_location=DEVICE))
    model.to(DEVICE).eval()
    adjacency = torch.load(artifact_dir / "adjacency.pt", map_location=DEVICE)
    return model, adjacency, saved_cfg


def predict_next_day(ticker: str, recent_panel_X: np.ndarray, recent_panel_S: np.ndarray,
                      artifact_dir: Path = ARTIFACT_DIR) -> dict:
    """recent_panel_X/S must already be the last `lookback` bars, shape (lookback, N, F)/(lookback, N, 1),
    with tickers in the same order as saved_cfg['tickers']. This is the function the Flask
    service in tradingskills.md §6 calls at inference time.
    """
    model, adjacency, saved_cfg = load_model_for_inference(artifact_dir)
    idx = saved_cfg["tickers"].index(ticker)
    x = torch.tensor(recent_panel_X, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    s = torch.tensor(recent_panel_S, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(x, s, adjacency)
    predicted_return = float(pred[0, idx].cpu())
    return {"ticker": ticker, "predicted_next_day_return": predicted_return}


# export_model(model, adjacency, cfg)
# predict_next_day("NVDA", X[-cfg.lookback:], S[-cfg.lookback:])

## 11. Caveats & integration notes

- **Supplementary signal, not a trigger.** This forecast is one more confluence input for Alpha's report card (`tradingskills.md` §10) — combine with the SMC/Liquidity/Composite skills, never act on it alone. Per §13, nothing here authorizes auto-execution.
- **Retrain periodically.** Sector relationships and regimes drift; a model trained once in 2025 will decay. Monthly or quarterly retraining is a reasonable starting cadence.
- **Lookahead bias is the main risk.** The chronological split above is necessary but not sufficient — double check that sentiment timestamps are strictly pre-prediction-time, and that any "sector" labels used to build the graph reflect what was known *at the time*, not today's classification.
- **The Finnhub headline corpus is cached, not live-refetched.** Re-running §4.1 after adding tickers or extending the date range only fetches what's missing — but if you need fresher headlines for a ticker already in the cache, delete its `data/finnhub_news_cache/<TICKER>.json` file first.
- **The sector-adjacency graph here is a simplification.** The paper also incorporates supply-chain relationships; reproducing that fully needs a curated relationship dataset (e.g. from a supply-chain data vendor), not just GICS sector membership.
- **The SSRN reference on real-time transformer forecasting couldn't be fetched** (blocked by Cloudflare bot-protection when this notebook was created). If you can get authenticated access, it's likely most relevant to turning this batch-trained notebook into a streaming/low-latency inference service — worth a follow-up pass once accessible.
- **Validate the paper's reported metrics independently** before repeating them anywhere — 0.80% MAPE / 25% earnings-window error reduction are as published, not something this notebook has reproduced yet.